# 🛒 Project 3 — E-commerce Product Intelligence

**Module 7 · Capstone · Session 7.7–7.8**

## The Problem
An e-commerce seller has hundreds of customer reviews. No human reads them all.
We build an AI pipeline that reads them and answers:
- Are customers happy or angry? → **Sentiment Agent**
- Can we trust these reviews, or are some fake/generic? → **Quality Agent**

## Architecture Decision: Batching
We do **NOT** call the LLM once per review. We send **10 reviews per call**
and get 10 labels back as JSON. This cuts 200 calls → 20 calls.

Why this matters (director-level point):
- **Rate limits stop being a problem** — 20 calls fits any free-tier quota.
- **Lower cost** — the instruction prompt is sent 20 times, not 200.
- Batching is the single biggest throughput lever in production LLM systems.

## Model Decision
`llama-3.3-70b-versatile` on Groq — the *strongest* free model, chosen for
**judgment quality**, not speed. Batching solves speed separately, so we
don't have to trade quality for throughput.

## Validation Decision
The Quality Agent outputs a business-critical number ("X% suspicious").
We validate it against **50 hand-labelled reviews** (our own ground truth),
and measure model reliability — not just "does it run".

## Stack
Groq (llama-3.3-70b-versatile) · pandas · batched JSON calls

## Session 7.7 — Step 1: Setup

Fresh Colab runtime, so we install and load everything up front:
- **`datasets`** — Hugging Face library to download the public reviews dataset.
- **`groq`** — the Groq API client for LLM calls.
- **`pandas`** — table handling (built into Colab, no install needed).

We also load API keys from Colab Secrets. Keeping all setup in one cell
means a single re-run fully recovers the environment after any restart.

In [1]:
# Install the two libraries Colab doesn't ship with.
# -q = quiet (less log spam). Rerun this cell if the runtime restarts.
!pip install -q datasets groq

# pandas: our table toolkit. datasets: to download the public dataset.
import pandas as pd
from datasets import load_dataset

# groq: the LLM client, same as Projects 1 & 2.
from groq import Groq

# Standard library helpers we'll use later.
import os        # to read environment variables (our API keys)
import time      # for pacing between calls if ever needed
import json      # to parse the JSON the LLM returns in batched calls

# Load the Groq API key from Colab Secrets into an environment variable.
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Create the Groq client once, reused everywhere.
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print("✅ Setup complete — libraries loaded, Groq client ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00
✅ Setup complete — libraries loaded, Groq client ready


## Session 7.7 — Step 2: Load & Sample 200 Reviews

Dataset: `amazon_polarity` (public, CC0 license — safe to use in a portfolio).
The full set is ~4M reviews, far too big. We take a reproducible sample of 200:

- Load only the smaller **test** split.
- **Shuffle with seed=42** — the raw file groups all negatives then all
  positives, so an unshuffled sample would be lopsided. A fixed seed makes
  it reproducible (an interviewer re-running the notebook gets the same 200).
- Convert to a pandas DataFrame for easy handling.

In [2]:
# Load ONLY the test split — avoids downloading the giant training split.
dataset = load_dataset("fancyzhx/amazon_polarity", split="test")

# Shuffle so our sample is a fair mix of positive & negative reviews.
# seed=42 = reproducible: same 200 rows every run.
dataset = dataset.shuffle(seed=42)

# Take the first 200 rows after shuffling.
sample = dataset.select(range(200))

# Convert the Hugging Face dataset into a pandas DataFrame (our table).
df = sample.to_pandas()

# Sanity checks: shape should be (200, 3); label counts should be roughly 50/50.
print("Shape (rows, columns):", df.shape)
print("\nLabel balance (0=negative, 1=positive):")
print(df["label"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

amazon_polarity/train-00002-of-00004.par(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

amazon_polarity/test-00000-of-00001.parq(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

Shape (rows, columns): (200, 3)

Label balance (0=negative, 1=positive):
label
0    103
1     97
Name: count, dtype: int64


## Session 7.7 — Step 3: Inspect & Add word_count

Quick data check before trusting anything (habit from Project 2):
confirm columns, check for missing values, and add a `word_count` column.

`word_count` matters later: the Quality Agent uses it as a safety net —
reviews too short to carry a real signal get flagged directly rather than
guessed at by the LLM.

In [3]:
# Confirm exact column names.
print("Columns:", df.columns.tolist())

# Check for missing values in each column.
print("\nMissing values per column:")
print(df.isnull().sum())

# Add a word_count column: split each review into words, count them.
df["word_count"] = df["content"].str.split().str.len()

# Quick look at the word_count spread.
print("\nWord count summary:")
print(df["word_count"].describe())

Columns: ['label', 'title', 'content']

Missing values per column:
label      0
title      0
content    0
dtype: int64

Word count summary:
count    200.000000
mean      73.430000
std       41.698789
min       12.000000
25%       39.000000
50%       65.000000
75%      106.000000
max      174.000000
Name: word_count, dtype: float64


## Session 7.7 — Step 4: The Batching Helper (reusable core)

The key architecture change: send 10 reviews per LLM call, get 10 labels
back as a JSON list. 200 reviews → 20 calls.

We build ONE reusable function that:
1. Takes a list of reviews + an instruction.
2. Numbers the reviews and asks the LLM to return a JSON list of labels.
3. Parses the JSON defensively (strip markdown fences, like Project 2).

This helper is agent-agnostic — we'll reuse it for BOTH the Sentiment
Agent and the Quality Agent by just changing the instruction. (DRY.)

In [4]:
def classify_batch(reviews, task_instruction, valid_labels):
    """
    Sends a batch of reviews to the LLM in ONE call.
    Returns a list of labels, one per review, in the same order.

    reviews         : list of review text strings
    task_instruction: what to classify (e.g. sentiment vs quality rules)
    valid_labels    : the allowed answers, e.g. ["positive","negative","neutral"]
    """
    # Number each review so the LLM can map answers back to inputs.
    numbered = "\n".join(f"{i+1}. {r}" for i, r in enumerate(reviews))

    prompt = f"""{task_instruction}

Allowed labels (use ONLY these): {valid_labels}

Reviews:
{numbered}

Return ONLY a JSON list of {len(reviews)} labels in order, like ["label1","label2",...].
No explanation, no markdown, just the JSON list."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",   # strongest free model, for judgment quality
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    raw = response.choices[0].message.content.strip()

    # Defensive parse: strip markdown code fences if the model added them.
    raw = raw.replace("```json", "").replace("```", "").strip()

    labels = json.loads(raw)  # turn the JSON text into a real Python list
    return labels

## Session 7.7 — Step 4 (test): One Batch of 10 Before Scaling

Batching's #1 failure mode: the LLM returns the wrong NUMBER of labels
(9 instead of 10) or wraps them in prose, breaking the review→label mapping.

So we test one batch of 10 first and verify:
1. It returned valid JSON (didn't crash the parse).
2. It returned exactly 10 labels (count matches input).
3. Every label is one of our allowed values.

In [5]:
# Grab the first 10 reviews as a test batch.
test_reviews = df["content"].head(10).tolist()

# Sentiment instruction — the task-specific part we pass into the reusable helper.
sentiment_instruction = "Classify the sentiment of each product review."
sentiment_labels = ["positive", "negative", "neutral"]

# Run ONE batched call.
result = classify_batch(test_reviews, sentiment_instruction, sentiment_labels)

# --- Verify the three things that matter ---
print("Number of labels returned:", len(result), "(expected 10)")
print("Labels:", result)

# Check every label is valid (catches typos / hallucinated labels).
invalid = [x for x in result if x not in sentiment_labels]
print("Invalid labels found:", invalid if invalid else "none ✅")

Number of labels returned: 10 (expected 10)
Labels: ['neutral', 'negative', 'negative', 'negative', 'negative', 'positive', 'neutral', 'positive', 'positive', 'positive']
Invalid labels found: none ✅


## Session 7.7 — Step 5: Run Sentiment on All 200 (Batched)

Loop through the 200 reviews in chunks of 10 → 20 calls total.

Guard: if any batch returns a label count that doesn't match its input
count, we record it as an error for that batch rather than silently
misaligning reviews with wrong labels (which would corrupt everything
downstream). This is the batched equivalent of "never trust output blindly."

In [6]:
def classify_all(reviews, instruction, valid_labels, batch_size=10):
    """
    Runs classify_batch over ALL reviews in chunks.
    Returns one label per review, in order.
    Guards against count mismatches per batch.
    """
    all_labels = []

    # Step through the list 10 at a time: 0-9, 10-19, 20-29, ...
    for start in range(0, len(reviews), batch_size):
        batch = reviews[start:start + batch_size]

        try:
            labels = classify_batch(batch, instruction, valid_labels)

            # Guard: did we get exactly one label per review in this batch?
            if len(labels) != len(batch):
                print(f"⚠️ Batch at {start}: expected {len(batch)}, got {len(labels)} — marking errors")
                labels = ["error"] * len(batch)

        except Exception as e:
            print(f"⚠️ Batch at {start} failed: {e} — marking errors")
            labels = ["error"] * len(batch)

        all_labels.extend(labels)
        print(f"Processed {start + len(batch)}/{len(reviews)}")

    return all_labels

# Run sentiment classification across all 200 reviews.
all_reviews = df["content"].tolist()
sentiment_results = classify_all(all_reviews, sentiment_instruction, sentiment_labels)

# Attach as a column and save.
df["agent_sentiment"] = sentiment_results
df.to_csv("clean_reviews_sample.csv", index=False)

print("\n✅ Done.")
print(df["agent_sentiment"].value_counts())

Processed 10/200
Processed 20/200
Processed 30/200
Processed 40/200
Processed 50/200
Processed 60/200
Processed 70/200
Processed 80/200
Processed 90/200
Processed 100/200
Processed 110/200
Processed 120/200
Processed 130/200
Processed 140/200
Processed 150/200
Processed 160/200
Processed 170/200
Processed 180/200
Processed 190/200
Processed 200/200

✅ Done.
agent_sentiment
negative    94
positive    86
neutral     20
Name: count, dtype: int64


## Session 7.7 — Step 6: Score Sentiment Against Dataset Labels

Map dataset labels (0/1) to words, compare to the agent's labels.
Neutral is excluded from scoring — the binary ground truth can't fairly
judge a review the agent flagged as genuinely mixed.

Note for the writeup: the stronger model produced 4× more neutrals (20 vs 5),
so more reviews are excluded here — a more honest, if lower-coverage, result.

In [7]:
# Map numeric labels to words for like-for-like comparison.
label_map = {1: "positive", 0: "negative"}
df["true_sentiment"] = df["label"].map(label_map)

# Score only on non-neutral reviews.
scored = df[df["agent_sentiment"] != "neutral"]
correct = (scored["agent_sentiment"] == scored["true_sentiment"]).sum()
total = len(scored)

print(f"Accuracy on {total} clear-cut reviews: {correct}/{total} = {correct/total:.1%}")
print(f"Excluded as neutral: {len(df) - total}/200")

df.to_csv("clean_reviews_sample.csv", index=False)

Accuracy on 180 clear-cut reviews: 178/180 = 98.9%
Excluded as neutral: 20/200


## Session 7.8 — Step 1: Quality Agent (Batched)

Same reusable batching helper, new instruction. The Quality Agent judges
*trustworthiness*, not sentiment:
- **trustworthy** — names a specific, checkable detail about using the product
- **suspicious** — generic praise/complaint that could apply to anything

Key instruction (learned from the earlier length-bias failure):
length does NOT decide the answer — a short review can be trustworthy.

We apply the <15-word safety net separately in code, not via the LLM.

In [8]:
# Quality instruction — note the explicit anti-length-bias line.
quality_instruction = """Classify each product review as trustworthy or suspicious.
Judge ONLY on whether it names a SPECIFIC, checkable detail about using this product.
Length does NOT decide the answer — a short review CAN be trustworthy.
'suspicious' = generic (e.g. "great product, fast shipping") with no real detail.
'trustworthy' = names a specific detail, even if short."""

quality_labels_allowed = ["trustworthy", "suspicious"]

# Run batched quality classification across all 200.
quality_results = classify_all(all_reviews, quality_instruction, quality_labels_allowed)
df["agent_quality"] = quality_results

# Apply the short-review safety net in CODE (not LLM):
# reviews under 15 words are labelled directly, overriding the LLM.
df.loc[df["word_count"] < 15, "agent_quality"] = "too short to judge"

df.to_csv("clean_reviews_sample.csv", index=False)

print("✅ Done.")
print(df["agent_quality"].value_counts())

Processed 10/200
Processed 20/200
Processed 30/200
Processed 40/200
Processed 50/200
Processed 60/200
Processed 70/200
Processed 80/200
Processed 90/200
Processed 100/200
Processed 110/200
Processed 120/200
Processed 130/200
Processed 140/200
Processed 150/200
Processed 160/200
Processed 170/200
Processed 180/200
Processed 190/200
Processed 200/200
✅ Done.
agent_quality
trustworthy           141
suspicious             58
too short to judge      1
Name: count, dtype: int64


## Session 7.8 — Step 2: Build Hand-Labelled Ground Truth (50 reviews)

Three models gave three different "% suspicious" answers (14% / 72% / 29%).
We cannot pick a winner without ground truth. So we create it: label 50
reviews BY HAND, then score the model against OUR labels.

This is what "directors fund annotation" means in practice — the number
is only as trustworthy as the labels you validate it against.

We label a reproducible sample of 50 (seed-fixed) so the exercise is
repeatable and honest, not cherry-picked.

In [9]:
# Take a reproducible 50-review sample to hand-label.
# random_state=42 = same 50 every time (reproducible, not cherry-picked).
label_sample = df.sample(50, random_state=42).copy()

# Create an empty column for YOUR human labels.
label_sample["human_quality"] = ""

# Save to a CSV you'll open and fill in by hand.
label_sample[["title", "content", "word_count", "agent_quality", "human_quality"]].to_csv(
    "to_label.csv", index=False
)

print("✅ Created to_label.csv with 50 reviews to label by hand.")
print("Columns: title, content, word_count, agent_quality (hidden from your judgment), human_quality (you fill this)")

✅ Created to_label.csv with 50 reviews to label by hand.
Columns: title, content, word_count, agent_quality (hidden from your judgment), human_quality (you fill this)


## Session 7.8 — Step 2: Label the 50 Reviews (in-notebook)

We loop through the 50 reviews one at a time. For each, you read it and type:
- `t` for trustworthy
- `s` for suspicious

The agent's guess is HIDDEN while you label (so it doesn't bias you).
Your answers are saved into the human_quality column as you go.

Reminder of the rule: specific checkable detail = trustworthy;
generic (could apply to any product) = suspicious. Length does NOT decide it.

In [10]:
# Load the file we just created.
to_label = pd.read_csv("to_label.csv")

# Loop through each review, ask for your judgment.
for i in range(len(to_label)):
    row = to_label.iloc[i]

    print("=" * 70)
    print(f"Review {i+1} of 50  (word count: {row['word_count']})")
    print("TITLE:", row["title"])
    print("\nCONTENT:", row["content"][:400])
    print("-" * 70)

    # Keep asking until we get a valid answer.
    while True:
        answer = input("Trustworthy or Suspicious? [t/s]: ").strip().lower()
        if answer in ["t", "s"]:
            break
        print("Please type just 't' or 's'.")

    # Save the human label ('t' -> trustworthy, 's' -> suspicious).
    to_label.loc[i, "human_quality"] = "trustworthy" if answer == "t" else "suspicious"

    # Save after EVERY label — so if Colab disconnects, you don't lose progress.
    to_label.to_csv("to_label.csv", index=False)

print("\n✅ All 50 labelled and saved to to_label.csv")

Review 1 of 50  (word count: 106)
TITLE: A hilarious how-to

CONTENT: What a gloriously funny book! Even the recipies were funny, and well, how funny did you think a recipie could be?! I "discovered" this book en route to Jamaica back in May--the stranger next to me read it all the way there. Well, the cover just grabbed me and I HAD to have it. It was a quick, light read that had a very wise and uplifting last chapter. Oh, and for those who are clueless like me in 
----------------------------------------------------------------------
Trustworthy or Suspicious? [t/s]: t


/tmp/ipykernel_841/1275790928.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'trustworthy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  to_label.loc[i, "human_quality"] = "trustworthy" if answer == "t" else "suspicious"


Review 2 of 50  (word count: 131)
TITLE: Monotonous, Implausible, Convoluted and Confusing

CONTENT: Let Me Call You Sweetheart, by Mary Higgins Clark, read by Bess Armstrong. A convoluted and seemingly implausible tale. Not MHC's best by a long shot, but may have lost a lot in the abridgement, which is pretty awful. Too many characters (and too many of them lawyers!) and the setting jumps between New York or New Jersey -- never quite sure. Finally, Bess Armstrong has a smooth, lovely voice, and 
----------------------------------------------------------------------
Trustworthy or Suspicious? [t/s]: t
Review 3 of 50  (word count: 40)
TITLE: The price of the product

CONTENT: I was looking to order this product but after reading the reviews there was a change of mind. A 6 pack would have been OK for the price but almost $50.00 for one Garnier is ridiculous. Is this price correct????
----------------------------------------------------------------------
Trustworthy or Suspicious? [t/s]: 

## Session 7.8 — Step 3: Score the Agent Against Human Labels

We now have ground truth: 50 human-labelled reviews. We compare the
agent's quality guesses to YOUR labels and compute:
- Overall accuracy (how often agent = human)
- WHERE it disagrees (agent said X, human said Y) — the useful part

Note: the human labels leaned trustworthy (44/50), so we look beyond
raw accuracy at the actual disagreement pattern.

In [11]:
# Load your labelled file and merge in the agent's guesses by matching content.
labelled = pd.read_csv("to_label.csv")

# Merge agent_quality from the main df onto the labelled 50, matching on content.
labelled = labelled.merge(
    df[["content", "agent_quality"]],
    on="content",
    how="left",
    suffixes=("", "_from_df")
)

# The agent_quality column is already in labelled from to_label.csv — use it directly.
# Compare agent vs human.
agree = (labelled["agent_quality"] == labelled["human_quality"]).sum()
total = len(labelled)
print(f"Agreement with human labels: {agree}/{total} = {agree/total:.1%}\n")

# Show the disagreements — this is where the insight is.
disagreements = labelled[labelled["agent_quality"] != labelled["human_quality"]]
print(f"Disagreements: {len(disagreements)}\n")
for _, row in disagreements.iterrows():
    print(f"AGENT: {row['agent_quality']:12} | HUMAN: {row['human_quality']:12} | {row['title']}")

Agreement with human labels: 36/50 = 72.0%

Disagreements: 14

AGENT: trustworthy  | HUMAN: suspicious   | Wheres "Do Whatcha Gotta Do"
AGENT: suspicious   | HUMAN: trustworthy  | Child Killer
AGENT: trustworthy  | HUMAN: suspicious   | "Device not recognized"
AGENT: suspicious   | HUMAN: trustworthy  | The Best Book I Ever Read
AGENT: suspicious   | HUMAN: trustworthy  | Go-to breakfast food
AGENT: suspicious   | HUMAN: trustworthy  | could have been alot better
AGENT: suspicious   | HUMAN: trustworthy  | Who would record let alone purchase?
AGENT: suspicious   | HUMAN: trustworthy  | Simply...is Phil, and is great.
AGENT: suspicious   | HUMAN: trustworthy  | One fire...
AGENT: suspicious   | HUMAN: trustworthy  | Not good
AGENT: suspicious   | HUMAN: trustworthy  | Buy this now!
AGENT: suspicious   | HUMAN: trustworthy  | I don't get it. This was totally waste of money.
AGENT: suspicious   | HUMAN: trustworthy  | Despicable, Disrespectful, Clueless
AGENT: suspicious   | HUMAN: trustw

## Session 7.8 — Step 5: Self-Consistency Check (do you agree with yourself?)

We re-label 8 reviews blind and compare to your ORIGINAL labels.
If a single human can't reproduce their own labels, then no model can
match them — proving the 72% ceiling is a task-specification problem,
not a model-quality problem. This is the annotation-reliability concept
directors fund (inter-annotator agreement, applied to yourself).

In [12]:
# Pull 8 reviews to re-label, using a DIFFERENT random seed so it's a fresh mix.
recheck = labelled.sample(8, random_state=7).reset_index(drop=True)

# Store your original labels, then blank them for the re-label.
recheck["original_label"] = recheck["human_quality"]

second_labels = []
for i in range(len(recheck)):
    row = recheck.iloc[i]
    print("=" * 70)
    print("TITLE:", row["title"])
    print("CONTENT:", row["content"][:350])
    print("-" * 70)
    while True:
        ans = input("Trustworthy or Suspicious? [t/s]: ").strip().lower()
        if ans in ["t", "s"]:
            break
        print("Please type 't' or 's'.")
    second_labels.append("trustworthy" if ans == "t" else "suspicious")

recheck["second_label"] = second_labels

# How often did you agree with YOURSELF?
self_agree = (recheck["original_label"] == recheck["second_label"]).sum()
print(f"\nSelf-consistency: {self_agree}/8 = {self_agree/8:.0%}")
print("\nWhere you changed your mind:")
print(recheck[recheck["original_label"] != recheck["second_label"]][["title", "original_label", "second_label"]])t


TITLE: Really terrible product
CONTENT: Well, i was so excited when i got the package. inserted the installation disk and i was prompted that installation was blocked due to compatibilty issues. I am running windows 7 ultimate and a microsoft product is incompatible with it. NO SOLUTION AVAILABLE CURRENTLY. What do I do? I am so very upset. Beware of this product.
----------------------------------------------------------------------
Trustworthy or Suspicious? [t/s]: t
TITLE: Good Album You Need This
CONTENT: I was really feeling this album. I think that they have some good features and some great tracks as well. I'm really feeling the guy with the high pitch voice, and the other guy is cool as well. The songs that stand out to me are Rap Life, Hip-Hop, and a couple others. The great thing about this album was seeing Cool Nutz from Portland on there, as
----------------------------------------------------------------------
Trustworthy or Suspicious? [t/s]: t
TITLE: Funny and silly
CONT

## Session 7.8 — Step 6: Cost & Scale Analysis (director view)

### Cost at industry scale (50M reviews/year, 2 agents = 100M LLM calls)
Assuming ~250 input + ~50 output tokens per call, at Groq list prices:

| Model | Annual inference cost |
|---|---|
| llama-3.1-8b-instant | ~$2,000 |
| gpt-oss-20b | ~$3,400 |
| llama-3.3-70b-versatile (used here) | ~$19,000 |

### The director insight
- Inference cost is NOT the binding constraint. Even the most expensive
  model is ~$19K/year — a fraction of one engineer's salary. Three
  engineers maintaining this pipeline cost ~$600K/year; inference is <5%
  of total cost of ownership.
- **Batching cut our calls 10× (200 → 20)** — eliminating the rate-limit
  problem AND cutting cost, with zero quality loss. This is the highest-
  leverage engineering decision in the project.
- Groq's **Batch API (50% off)** and **prompt caching** (cached tokens
  don't count toward limits) would cut cost further — review analysis is
  inherently batch, so real-time pricing is the wrong model to optimize.

### The real constraint
Not cost. Not latency. **Correctness on subjective tasks.** The money
saved by picking a cheap model is trivial next to the business cost of
shipping a wrong "% fake reviews" number to sellers (churn, legal exposure,
trust damage). A director optimizes the expensive variable, not the cheap one.